Criar database Bronze:

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
print("Database bronze criado com sucesso")

Database bronze criado com sucesso


Criar tabelas a partir de cada CSV:

In [0]:
#aqui vou criar uma lista com nome dos arquivos Csv's
import os
caminhoArquivos = "/Volumes/workspace/schemavisagio/olist_raw_data"
arquivosCsv = os.listdir(caminhoArquivos)
arquivosCsv

['olist_order_reviews_dataset.csv',
 'olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [0]:
from pyspark.sql.functions import current_timestamp

for arquivo in arquivosCsv:
    caminhoCompleto = os.path.join(caminhoArquivos, arquivo)

    df = spark.read \
        .option("header", True) \
        .option("sep", ",") \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("multiLine", True) \
        .csv(caminhoCompleto)

    df = df.withColumn("timestamp_ingestion", current_timestamp())

    #aqui estou pegando so a indicação do arquivo para usar no titulo da tabela:
    nomeArquivo = arquivo.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")

    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"bronze.tb_{nomeArquivo}")

    print(f"Tabela bronze.tb_{nomeArquivo} criada")

Tabela bronze.tb_order_reviews criada
Tabela bronze.tb_customers criada
Tabela bronze.tb_geolocation criada
Tabela bronze.tb_order_items criada
Tabela bronze.tb_order_payments criada
Tabela bronze.tb_orders criada
Tabela bronze.tb_products criada
Tabela bronze.tb_sellers criada
Tabela bronze.tb_product_category_name_translation criada


Ingestão de API (cotação do dolar)

In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, to_timestamp

#pra essa ingestao vou usar as datas que tem nas info do dataset no kaggle:
data_inicio_formatada = "01-01-2016"
data_fim_formatada = "12-31-2018"

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(url)
data_json = response.json()
valores = data_json['value']

#converter para dataframe sparks:
rows = [Row(dataHoraCotacao=d['dataHoraCotacao'], cotacaoCompra=float(d['cotacaoCompra'])) for d in valores]
        
df_dolar = spark.createDataFrame(rows)
df_dolar = df_dolar.withColumn("timestamp_ingestion", current_timestamp())

df_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("bronze.tb_cotacao_dolar")

print("Tabela bronze.tb_cotacao_dolar criada")





Tabela bronze.tb_cotacao_dolar criada
